In [ ]:
# %pip install numpy pandas matplotlib seaborn adjustText
# or use uv
# %uv add numpy pandas matplotlib seaborn adjustText

In [2]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, NullLocator, ScalarFormatter
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle, Circle
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import matplotlib.font_manager as fm
from io import StringIO
import os

# Set global style
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.dpi'] = 300

# find font Linux Libertine
# font_path = r'C:\Users\xxxx\AppData\Local\Microsoft\Windows\Fonts\LinLibertine_Rah.ttf'
# font = fm.FontProperties(fname=font_path, size=8)
# fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'Linux Libertine'
plt.rcParams['mathtext.fontset'] = 'cm'
# Color palette
COLORS = {
    'red': '#E74C3C',
    'blue': '#3498DB',
    'green': '#2ECC71',
    'lime': "#9ACE38",
    'orange': '#E67E22',
    'purple': '#9B59B6',
    'gray': '#7F8C8D',
    'dark': '#2C3E50',
    'light_red': '#FADBD8',
    'light_blue': '#D4E6F1',
    'light_green': '#D5F4E6',
    'light_orange': '#FAE5D3',
    'light_purple': '#E8DAEF',
}

## Pareto Frontier

In [7]:
def create_efficiency_comparison():
    """Figure 5: Efficiency Comparison"""
    print("Creating efficiency_comparison.pdf...")
    
    # Data
    methods = ['ES*', 'Rand', 'TPOT', 'Auto-sklearn', 'CtxPipe', 
               'HAI-AI', 'LOO', 'MC-SP', 'H-SP',
               'SAGA', 'L2C', 'DP']
                # ES    Rand   TPOT   AUSK   CtxPipe  HAI  LOO     MC-SP   H-SP    SAGA   L2C     DP
    accuracy = [0.862, 0.762,  0.769, 0.738, 0.802, 0.774, 0.821,  0.839,  0.847,  0.754, 0.807,  0.784]
    time_sec = [22680,  48.2,  342.8, 482.9, 102.4,  89.2, 261.8, 2035.8, 1294.3, 1384.5,  97.6, 1029.8]
    
    fig, ax1 = plt.subplots(figsize=(5.5, 3.3))
    
    # Left plot: Accuracy vs Time (log scale)
    colors = [COLORS['gray'], COLORS['gray'], COLORS['gray'], COLORS['gray'],
              COLORS['blue'], COLORS['blue'], COLORS['orange'], COLORS['purple'], COLORS['red'], 
              COLORS['gray'], COLORS['blue'], COLORS['gray']]
    
    for i, (m, t, a) in enumerate(zip(methods, time_sec, accuracy)):
        if m == 'H-SP':
            ax1.scatter(t, a, s=250, c=colors[i], marker='*', zorder=5, 
                       edgecolors='darkred', linewidths=1.5)
        elif 'ES*' in m:
            ax1.scatter(t, a, s=150, c=colors[i], marker='D', alpha=0.6, zorder=3)
        else:
            ax1.scatter(t, a, s=100, c=colors[i], marker='o', alpha=0.7, zorder=3)
        
        # Add labels
        offset = (10, -3) # if t < 1000 else (-40, 5)
        if m == 'H-ShapleyPipe':
            offset = (-25, -3)
        elif m == 'L2C':
            offset = (10, 1)
        
        ax1.annotate(m.replace('\n', ' '), (t, a), xytext=offset, 
                    textcoords='offset points', fontsize=11, alpha=0.9)
    
    # Add Pareto frontier
    pareto_x = [48.2, 97.6, 1294.3]
    pareto_y = [0.762, 0.807, 0.847]
    ax1.plot(pareto_x, pareto_y, '--', color=COLORS['green'], alpha=0.5, linewidth=2)
    
    # Highlight ShapleyPipe region
    ax1.axhspan(0.835, 0.87, alpha=0.1, color=COLORS['red'])
    ax1.axvspan(50, 1500, alpha=0.05, color=COLORS['green'])
    
    ax1.set_xlabel('Running Time (seconds, log scale)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Average Accuracy', fontsize=12, fontweight='bold')
    # ax1.set_title('(a) Efficiency-Effectiveness Trade-off', fontsize=12, fontweight='bold')
    ax1.set_xscale('log')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(30, 100000)
    ax1.set_ylim(0.70, 0.88)
    
    # Custom legend
    legend_elements = [Line2D([0], [0], linestyle='--', color=COLORS['green'],
               label='Pareto Frontier')]
    ax1.legend(handles=legend_elements, loc='lower right', fontsize=12)
    
    plt.tight_layout()
    plt.savefig('saves/efficiency_comparison_2.pdf', dpi=300, bbox_inches='tight')
    plt.close()
    print("  Done!")
    
create_efficiency_comparison()

Creating efficiency_comparison.pdf...
  Done!


## Positional Sensitivity v.s. Gain over Baseline

In [6]:
from matplotlib.ticker import MultipleLocator
from adjustText import adjust_text
def draw_ax(ax, all_data, xlabel, ylabel, title, threshold_y_shift=0, draw_legend=False):
    dist_median = all_data['dist'].median()
    high_dist_data = all_data[all_data['dist'] > dist_median]
    low_dist_data = all_data[all_data['dist'] <= dist_median]
    all_distances = []
    all_improvements = []
    all_names = []
    all_categories = []
    for t in high_dist_data.itertuples():
        all_distances.append(t.dist)
        all_improvements.append((t.sv - t.base) * 100)
        all_names.append(t.name)
        all_categories.append('high')
    for t in low_dist_data.itertuples():
        all_distances.append(t.dist)
        all_improvements.append((t.sv - t.base) * 100)
        all_names.append(t.name)
        all_categories.append('low')
    all_distances = np.array(all_distances)
    all_improvements = np.array(all_improvements)
    high_mask = np.array(all_categories) == 'high'
    low_mask = np.array(all_categories) == 'low'
    high_sc = ax.scatter(all_distances[high_mask], all_improvements[high_mask],
               c=COLORS['red'], s=60, alpha=0.8, label='High-Sensitivity',
               edgecolors='darkred', linewidths=1, marker='o', zorder=3)
    low_sc = ax.scatter(all_distances[low_mask], all_improvements[low_mask],
               c=COLORS['blue'], s=60, alpha=0.8, label='Low-Sensitivity',
               edgecolors='darkblue', linewidths=1, marker='s', zorder=3)
    slope, intercept, r_value, p_value, std_err = stats.linregress(all_distances, all_improvements)
    x_line = np.linspace(min(all_distances) - 0.3, max(all_distances) + 0.3, 100)
    y_line = slope * x_line + intercept
    trend_line = ax.plot(x_line, y_line, 'k--', linewidth=1.5, alpha=0.6,
            label='Trend line', zorder=2)[0]
    ax.axvline(x=dist_median, color='gray', linestyle=':', linewidth=1.5, alpha=0.7, zorder=1)
    ax.text(dist_median, max(all_improvements)+threshold_y_shift, f'Threshold ({dist_median:.3f})', fontsize=11, color='gray', ha='left', va='top')
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.8, alpha=0.5, zorder=1)
    high_mean = np.mean(all_improvements[high_mask])
    low_mean = np.mean(all_improvements[low_mask])
    ax.fill_between([np.floor(min(all_data['dist'])), dist_median], [low_mean, low_mean], alpha=0.1, color=COLORS['blue'], zorder=0)
    ax.fill_between([dist_median, 0.2], [high_mean, high_mean], alpha=0.1, color=COLORS['red'], zorder=0)
    ax.hlines(high_mean, dist_median, 0.2, colors=COLORS['red'], linestyles='--', linewidth=1.5, alpha=0.6)
    ax.hlines(low_mean, np.floor(min(all_data['dist'])), dist_median, colors=COLORS['blue'], linestyles='--', linewidth=1.5, alpha=0.6)
    ax.text(0.2-0.002, high_mean + np.sign(high_mean) * 10, f'Avg: {high_mean:+.1f}%', fontsize=11, color=COLORS['red'],
            fontweight='bold', ha='right', va='center')
    ax.text(np.floor(min(all_data['dist']))+0.002, low_mean + np.sign(low_mean)*17, f'Avg: {low_mean:+.1f}%', fontsize=11, color=COLORS['blue'],
            fontweight='bold', ha='left', va='center')
    ax.set_xlabel(xlabel, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.xaxis.set_major_locator(MultipleLocator(0.05))
    ax.xaxis.set_major_formatter('{x:.2f}')
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    ax.set_xlim(np.floor(min(all_data['dist'])), 0.2)
    ax.set_ylim(-2, 35)
    ax.set_title(title)
    if draw_legend:
        ax.legend(loc='upper right', fontsize=10, framealpha=0.95, edgecolor='gray')

    # Annotate each data point — use adjustText to auto-avoid overlap
    texts = []
    orig_positions = []
    for i, name in enumerate(all_names):
        orig_positions.append((all_distances[i], all_improvements[i]))
        t = ax.text(all_distances[i], all_improvements[i], name,
                    fontsize=7, alpha=0.9, color='#2C3E50',
                    ha='center', va='center')
        texts.append(t)
    adjust_text(texts, ax=ax,
                force_text=(1.8, 2.0), force_points=(0.2, 0.2),
                expand=(1.5, 1.6), precision=0.01,
                lim=300, arrowprops=None)

    # Draw fine connecting lines from adjusted positions back to original points
    for t, (x0, y0) in zip(texts, orig_positions):
        pos = t.get_position()
        if abs(pos[0] - x0) > 0.003 or abs(pos[1] - y0) > 0.3:
            ax.plot([x0, pos[0]], [y0, pos[1]], color='gray', linewidth=0.4,
                    alpha=0.3, zorder=1)

    return [high_sc, low_sc, trend_line], ['High-Distance', 'Low-Distance', 'Trend line']


all_data_mean_acc_gain_vs_prob_str = '''name	dist	base	sv
abalone	0.077182954	0.268	0.278
ada_prior	0.030543477	0.821	0.845
avila	0.075827053	0.638	0.941
connect-4	0.028989387	0.723	0.794
eeg	0.067294464	0.675	0.864
google	0.01947548	0.600	0.665
house	0.071124743	0.914	0.938
jungle_chess	0.048506642	0.747	0.847
micro	0.029998946	0.589	0.643
mozilla4	0.04594222	0.907	0.934
obesity	0.117121449	0.807	0.901
page-blocks	0.015009941	0.945	0.970
pbcseq	0.060535659	0.748	0.784
pol	0.056199436	0.911	0.982
run_or_walk	0.055973117	0.867	0.984
shuttle	0.026587081	0.886	1.000
uscensus	0.023284975	0.831	0.846
wall-robot	0.053375797	0.849	0.952
cmc	0.155172856	0.535	0.615
diabetes	0.065891456	0.789	0.870
jm1	0.021010064	0.812	0.824
kc1	0.023755158	0.866	0.882
kc2	0.041687258	0.831	0.868
pc4	0.021796714	0.910	0.938
phoneme	0.048271981	0.799	0.882
qsar-biodeg	0.081234947	0.868	0.906
vehicle	0.165425785	0.800	0.906
'''

all_data_mean_acc_gain_vs_prob = pd.read_csv(StringIO(all_data_mean_acc_gain_vs_prob_str), sep='\t')

def draw_prob():
    fig, ax = plt.subplots(1,1,figsize=(5.5, 3))
    h1, l1 = draw_ax(ax, all_data_mean_acc_gain_vs_prob, 
                    'Positional Sensitivity', 
                    'Accuracy Gain over Baselines (%)', 
                    None,
                    threshold_y_shift=-2,
                    draw_legend=True)
    # fig.legend(h1, l1, loc='lower center', ncol=3, fontsize=10,
    #         framealpha=0.95, edgecolor='gray', bbox_to_anchor=(0.5, 0))
    # plt.tight_layout(rect=(0, 0.08, 1, 1))
    plt.savefig('saves/pos_sensitivity_vs_mean_acc_gain.pdf', dpi=100, bbox_inches='tight')
    # plt.savefig('./pos_sensitivity_vs_mean_acc_gain.png', dpi=300, bbox_inches='tight')
    plt.close()

draw_prob()


## Sensitivity Analysis

In [10]:
def merged_mt():
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(6, 6))

    # Extended data for visualization (including smaller M)
    m_extended = [2, 3, 4, 5, 6, 7, 8, 9]
    acc_extended = np.array([0.798, 0.815, 0.832, 0.839, 0.847, 0.843, 0.840, 0.808])
    ax1.plot(m_extended, acc_extended, 'o-', color=COLORS['blue'], linewidth=2.5, 
             markersize=10, markerfacecolor='white', markeredgewidth=2, label='Accuracy')
    optimal_idx = np.argmax(acc_extended)
    ax1.scatter([m_extended[optimal_idx]], [acc_extended[optimal_idx]], 
                s=200, c=COLORS['green'], marker='*', zorder=5, 
                edgecolors='darkgreen', linewidths=1.5, label=f'Optimal ($\\overline{{M}}$={m_extended[optimal_idx]})')
    
    # Add shaded region for recommended range
    ax1.axvspan(5, 7, alpha=0.15, color=COLORS['green'], label='Recommended Range')
    
    # Add overfitting annotation
    ax1.annotate('Over-transformation\nRegion', xy=(5.6, 0.79), fontsize=11, color=COLORS['red'],
                ha='center')
    ax1.annotate('', xy=(9, 0.808), xytext=(8.2, 0.8),
                arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1.5))
    
    ax1.set_xlabel('Maximum Pipeline Length $\\overline{M}$', fontsize=12)
    ax1.set_ylabel('Average Accuracy', fontsize=12)
    ax1.set_title('(a) Accuracy vs. Maximum Pipeline Length', fontsize=12, x=0.4)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left', fontsize=10)
    ax1.set_xlim(1.5, 9.5)
    ax1.set_ylim(0.775, 0.925)
    ax1.set_xticks(m_extended)
    
    # Right plot: Evaluations and Time vs M
    ax2_twin = ax2.twinx()
    
    m_values = [5, 6, 7, 8, 9]
    eval_count = [1032, 2136, 4640, 6184, 8560]
    time_sec =   [543, 1294, 3308, 5175, 6397]
    
    line1 = ax2.bar(np.array(m_values) - 0.2, np.array(eval_count) / 1000, 0.35, 
                    color=COLORS['blue'], alpha=0.7, label='Evaluations (×1000)')
    line2 = ax2_twin.bar(np.array(m_values) + 0.2, np.array(time_sec) / 60, 0.35, 
                         color=COLORS['orange'], alpha=0.7, label='Time (min)')
    
    ax2.set_xlabel('Maximum Pipeline Length $\\overline{M}$', fontsize=12)
    ax2.set_ylabel('Number of Evaluations (×1000)', fontsize=12, color=COLORS['blue'])
    ax2_twin.set_ylabel('Running Time (minutes)', fontsize=12, color=COLORS['orange'])
    # ax2.set_title('(b) Computational Cost vs. Pipeline Length', fontsize=12)
    ax2.set_title('(b) Cost vs. Maximum Pipeline Length', fontsize=12, x=0.6)
    
    ax2.tick_params(axis='y', labelcolor=COLORS['blue'])
    ax2_twin.tick_params(axis='y', labelcolor=COLORS['orange'])
    
    # Combined legend
    lines1, labels1 = ax2.get_legend_handles_labels()
    lines2, labels2 = ax2_twin.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)
    
    ax2.set_xticks(m_values)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # ax3
    t_values = [25, 50, 75, 100, 150, 200, 300, 500]
    accuracy = [0.832, 0.843, 0.847, 0.846, 0.8475, 0.847, 0.8475, 0.8478]
    std_dev = [0.018, 0.015, 0.009, 0.010, 0.009, 0.007, 0.007, 0.006]
    ax3.errorbar(t_values, accuracy, yerr=std_dev, fmt='o-', color=COLORS['blue'], 
                 linewidth=2, markersize=6, capsize=4, capthick=2,
                 markerfacecolor='white', markeredgewidth=2, label='Accuracy ± Std')
    
    # Fill between for confidence
    acc_upper = np.array(accuracy) + np.array(std_dev)
    acc_lower = np.array(accuracy) - np.array(std_dev)
    ax3.fill_between(t_values, acc_lower, acc_upper, alpha=0.2, color=COLORS['blue'])
    
    # Add convergence annotation
    ax3.axhline(y=0.847, color=COLORS['green'], linestyle='--', alpha=0.7, label='Converged Value')
    ax3.annotate('Convergence\n(~0.847)', xy=(300, 0.839), fontsize=11, 
                ha='center', va='bottom', bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'))
    
    # Mark recommended point
    ax3.axvline(x=75, color=COLORS['orange'], linestyle=':', linewidth=2, alpha=0.7)
    ax3.text(85, 0.821, 'Recommended\n($n_\\text{sample}$=75)', fontsize=11)
    
    ax3.set_xlabel('Number of Monte Carlo Samples $n_\\text{sample}$', fontsize=12)
    ax3.set_ylabel('Average Accuracy', fontsize=12)
    ax3.set_title('(c) Accuracy Convergence with MC Samples', fontsize=12, x=0.45)
    ax3.legend(loc='lower right', fontsize=10)
    ax3.grid(True, alpha=0.3)
    ax3.set_xscale('log')
    ax3.set_xticks(     [20,  30,  40,  50,  60,  70,  80,  90,  100, 200, 300, 400, 500, 600])
    ax3.set_xticklabels(['20','30',None,None,None,None,None,None,'100',None,'300',None,None,'600'])
    ax3.xaxis.set_minor_locator(NullLocator()) 
    # ax3.xaxis.set_major_formatter(ScalarFormatter())
    ax3.set_xlim(20, 600)
    ax3.set_ylim(0.80, 0.86)
    
    mab_cnt =    [ 20,    50,    100,    200,     300,   500,   1000,   1500,   2000,   3000, 5000   ]
    mab_values = [ 0.831, 0.839, 0.8382, 0.843, 0.841, 0.845, 0.8467, 0.8461, 0.8463, 0.8451, 0.8467 ]
    mab_std =    [ 0.015, 0.016, 0.0153, 0.014, 0.013, 0.010,  0.010,  0.011,  0.009,  0.012, 0.010  ]

    ax4.errorbar(mab_cnt, mab_values, yerr=mab_std, fmt='o-', color=COLORS['blue'], 
                 linewidth=2, markersize=6, capsize=4, capthick=2,
                 markerfacecolor='white', markeredgewidth=2, label='Accuracy ± Std')

    ax4.fill_between(mab_cnt, np.array(mab_values) - np.array(mab_std), 
                     np.array(mab_values) + np.array(mab_std), alpha=0.2, color=COLORS['blue'])
    ax4.set_xlabel('Number of MAB Samples $T_\\text{mab}$', fontsize=12)
    # ax4.set_ylabel('Average Accuracy', fontsize=12, fontweight='bold')
    ax4.set_title('(d) Accuracy Convergence with MAB', fontsize=12, loc='left', horizontalalignment='left')
    ax4.legend(loc='lower right', fontsize=11)
    ax4.grid(True, alpha=0.3)
    ax4.set_xscale('log')
    ax4.set_xlim(10, 6000)
    ax4.set_ylim(0.80, 0.86)

    ax4.axvline(x=1000, color=COLORS['orange'], linestyle=':', linewidth=2, alpha=0.7)
    ax4.text(70, 0.812, 'Recommended\n($T_\\text{mab}$=1000)', fontsize=11)

    plt.tight_layout()
    plt.savefig('saves/merged_mt.pdf', dpi=300, bbox_inches='tight')

    plt.close()

    print("  Done!")

merged_mt()

  Done!
